In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=2ω_a# Drive frequency
ω_p=2ω_a-ω_d

ϵ_p = 0.7
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2

buffer_phase = 0.0
pump_phase = 0.0

κ_b = pi/10.4;

In [ ]:
#ATS + Mean_Field
function equation_1ODE_bibuffer!(du,u,p::SimulationParams{SystemATS_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    ϕ_a = u[1]
    ϕ_b_1 = u[2]
    ϕ_b_2 = u[3]
    ϕ_p_a = u[4]
    ϕ_p_b_1 = u[5]
    ϕ_p_b_2 = u[6]

    #coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_1 * sqrt(Z_b)/ϕ_quantum + ϕ_b_2 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_1 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling_2 = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_2 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)/2#/sqrt(2)
        
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b_1
    du[3] = ϕ_p_b_2
    du[4] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*(coupling+coupling_2)) - 2 * p.environment_prepared.κ_a * du[1]
    du[5] = -ω_b*(ω_b*ϕ_b_1 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling) - 2 * p.environment_prepared.κ_b * du[2]
    du[6] = -ω_b*(ω_b*ϕ_b_2 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling_2) - 2 * p.environment_prepared.κ_b * du[3]
    return nothing
end

#ATS + Mean_Field
function equation_1ODE_tribuffer!(du,u,p::SimulationParams{SystemATS_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    ϕ_a = u[1]
    ϕ_b_1 = u[2]
    ϕ_b_2 = u[3]
    ϕ_b_3 = u[4]
    ϕ_p_a = u[5]
    ϕ_p_b_1 = u[6]
    ϕ_p_b_2 = u[7]
    ϕ_p_b_3 = u[8]

    #coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + (ϕ_b_1 + ϕ_b_2 + ϕ_b_3) * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_1 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling_2 = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_2 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling_3 = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_3 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)/3#/sqrt(3)
        
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b_1
    du[3] = ϕ_p_b_2
    du[4] = ϕ_p_b_3
    du[5] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*(coupling+coupling_2+coupling_3)) - 2 * p.environment_prepared.κ_a * du[1]
    du[6] = -ω_b*(ω_b*ϕ_b_1 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling) - 2 * p.environment_prepared.κ_b * du[2]
    du[7] = -ω_b*(ω_b*ϕ_b_2 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling_2) - 2 * p.environment_prepared.κ_b * du[3]
    du[8] = -ω_b*(ω_b*ϕ_b_3 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling_3) - 2 * p.environment_prepared.κ_b * du[4]
    return nothing
end

In [ ]:
t_max = 30/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(0.0, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
ϵ_p_list = vcat(0.2:.01:1.5);
ϵ_d_g2_ratio_list = [10.12];

In [ ]:
using ChaosTools, ProgressBars

In [ ]:
 
Lyap_coef_chaos = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
Lyap_coef_chaos_bibuffer = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
Lyap_coef_chaos_tribuffer = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_p_list))
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 15/κ_b
for i in ProgressBar(1:length(ϵ_p_list))
    ϵ_p_i = ϵ_p_list[i]
    drive.ϵ_0 = ϵ_p_i
    #g2_i = besselj(1,ϵ_p_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    g2_i = sin(ϵ_p_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_i
        drive.ϵ_d = ϵ_d_j
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        problem = CoupledODEs(equation_1ODE!, u_0_new, p)
        lyap_1 = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        Lyap_coef_chaos[j,i] = lyap_1
        
        drive.ϵ_d = ϵ_d_j#/sqrt(2)
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        problem = CoupledODEs(equation_1ODE_bibuffer!, u_0_new, p)
        lyap_1 = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        Lyap_coef_chaos_bibuffer[j,i] = lyap_1
        
        drive.ϵ_d = ϵ_d_j#/sqrt(3)
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        problem = CoupledODEs(equation_1ODE_tribuffer!, u_0_new, p)
        lyap_1 = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        Lyap_coef_chaos_tribuffer[j,i] = lyap_1
    end
end

In [ ]:
g2_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b/2π*1e3
plot(g2_list,transpose(Lyap_coef_chaos), label = "Mono-buffer")
plot!(g2_list,transpose(Lyap_coef_chaos_bibuffer), label = "Bi-buffer")
plot!(g2_list,transpose(Lyap_coef_chaos_tribuffer), label = "Tri-buffer")
plot!(xlabel=L"g_2^\mathrm{eff}/2\pi"*" (MHz)", ylabel=L"$\lambda$")

In [ ]:
g2_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b/2π*1e3
plot(g2_list,transpose(Lyap_coef_chaos), label = "Mono-buffer")
plot!(g2_list,transpose(Lyap_coef_chaos_bibuffer), label = "Bi-buffer")
plot!(g2_list,transpose(Lyap_coef_chaos_tribuffer), label = "Tri-buffer")
plot!(xlabel=L"g_2^\mathrm{eff}/2\pi"*" (MHz)", ylabel=L"$\lambda$")